In [1]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from datasets import load_dataset

ds = load_dataset("PotatoHD/ru-text-corpus")['train'].to_pandas()
ds = ds[ds.src == 'news'].copy()

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/datasets/PotatoHD/ru-text-corpus/resolve/314daff159baa2c04d6992bd99c019966013effc/ru-text-corpus.py
Retrying in 1s [Retry 1/5].


In [3]:
ds.text.str.len().sum()

np.int64(191773773)

In [4]:
import re
import pandas as pd

def to_lower(text):
    return str(text).lower() if pd.notna(text) else text


def remove_punct(text):
    return re.sub(r"[^\w\s]", " ", str(text), flags=re.UNICODE) if pd.notna(text) else text


def to_tokens(text):
    return re.findall(r"\w+", str(text), flags=re.UNICODE) if pd.notna(text) else []

def tokenize(text):
    return to_tokens(remove_punct(to_lower(text)))

In [5]:
ds['tokens'] = ds.text.apply(to_lower).apply(remove_punct).apply(to_tokens)
ds.head(5)

,text,src,tokens
17177,Сегодня транспортный налог начисляется в завис...,news,"[сегодня, транспортный, налог, начисляется, в,..."
17178,Словосочетание «музыкальный кинофестиваль» уже...,news,"[словосочетание, музыкальный, кинофестиваль, у..."
17179,После более чем 12-часовых консультаций Совет ...,news,"[после, более, чем, 12, часовых, консультаций,..."
17180,"Жертвами урагана «Агата», обрушившегося на Цен...",news,"[жертвами, урагана, агата, обрушившегося, на, ..."
17181,Решение ограничить рост тарифов естественных м...,news,"[решение, ограничить, рост, тарифов, естествен..."


In [6]:
#Статистика: число слов, размер словаря, двадцать самых частых слов.
print("Число слов", ds.tokens.str.len().sum())
print("Размер словаря", len(ds.tokens.explode().unique()))
print("Двадцать самых частых слов.", ds.tokens.explode().value_counts().head(20))

Число слов 26805885
Размер словаря 508506
Двадцать самых частых слов. tokens
в      1147476
и       677850
на      504462
не      344342
что     307427
с       305267
по      247341
а       142979
из      137719
за      133083
к       118637
как     111675
но      109778
о       104261
это      99392
для      96082
он       95534
его      86559
от       85971
у        79842
Name: count, dtype: int64


In [7]:
ds_train = ds.sample(frac=0.9)
ds_test = ds.drop(ds_train.index)

del ds #освобождаю ресурсы для экономии оперативки

In [8]:
import random 
import numpy as np 

class CounterLM:
    
    def __init__(self, n=2):
        self.prob_table = None
        self.words_freq = None
        self.n = n
        self.V = 0
        

    @staticmethod
    def to_ngram(tokens, n=2):
        tokens_len = len(tokens)
        return [tuple(tokens[i:i+n]) for i in range( tokens_len-n+1)]


    def fit(self, text_df, tokens):
        prob_table = text_df[tokens].apply(self.to_ngram).explode().value_counts().rename("c_vw").to_frame()
        self.words_freq = text_df[tokens].explode().value_counts()
        self.V = text_df[tokens].explode().nunique()

        prob_table[["v", "w"]] = pd.DataFrame(prob_table.index.tolist(), index=prob_table.index)
        prob_table["c_v"]  = prob_table["v"].map(self.words_freq)
        prob_table["prob"] = prob_table["c_vw"] / prob_table["c_v"]

        self.prob_table = prob_table.reset_index()

        

    def prob(self, v, w):
        prob = self.prob_table[(self.prob_table.v == v) & (self.prob_table.w == w)].prob
        if prob.shape[0] > 0:
            return prob.iloc[0]
        else:
            return 0.0

    def generate(self, start, output_len=30, T=1):
        output_text = [start]
        for i in range(output_len):
            #print(output_text)
            probs = self.prob_table[self.prob_table.v == output_text[-1]]
            tokens = probs.w.tolist()
            wth = probs.prob.pow(1.0/T).tolist()

            output_text += random.choices(tokens, wth)
        return ' '.join(output_text)

    def logprob(self, text, alpha=1):
        logP = 0.0

        for i in range(len(text) - self.n + 1):
            v,w = text[i:i+self.n]

            c_vw = self.prob_table[(self.prob_table.v == v) & (self.prob_table.w == w)].c_vw
            if c_vw.shape[0] > 0:
                c_vw = c_vw.iloc[0]
            else:
                c_vw = 0.0
            c_v = self.words_freq['v'] 

            logP += np.log((c_vw+alpha) / (c_v + alpha * self.V))

        return logP
        


In [9]:
"""
Сглаживание лапласа помогает избавится от случаев с -inf оценкой
 правдоподобия для биграм, которых не было в датасете. Если 
попадается хотя-бы одна такая биграма то вся оценка правдоподобия будет -inf, 
а две оценки с -inf не сравнимы. Сглаживание помогает "прикинуть" 
редкость для пары который нет на основе размера словаря. 
"""


'\nСглаживание лапласа помогает избавится от случаев с -inf оценкой\n правдоподобия для биграм, которых не было в датасете. Если \nпопадается хотя-бы одна такая биграма то вся оценка правдоподобия будет -inf, \nа две оценки с -inf не сравнимы. Сглаживание помогает "прикинуть" \nредкость для пары который нет на основе размера словаря. \n'

In [10]:
lm = CounterLM(n=2)

lm.fit(ds_train, tokens='tokens')



In [11]:
for i in range(5):
    print(lm.generate(start='анекдот', T=1.5))

анекдот на переполненной донбасс белорусские сми опубликовал заявление доку умаровым на предполагаемые хулиганы дрались и новотроицкое руководству renault 6 50 мы не повторится против клуба не носит негативного рейтингового агентства sana
анекдот о 260 получили шквал атак поведя со злом и курляндского затея остроумная и готовы принять решение о бешеном темпе пойдет снег в полотно предназначено исключительно маркетинговые технологии htc опровергла сама
анекдот и удивительную веху для планшетных компьютеров теперь получили 573 млн меньше срочных дебатов пожалуй имеет кадровые решения нужна продолжал сражаться за преимущество из большой экономической оценки знаменитого фотографа а сравнимый
анекдот состоялся заключительный этап испытания обеих стран еврозоны заявило американское оружие и большинства подмосковной команды греции также штрафовать в сфере питания одежды рулон из прокуратуры суда стало ясно показывает рекордную отметку
анекдот времен солидный бразильский 7 образование самостоя

In [12]:
for i in range(5):
    print(lm.generate(start='анекдот', T=0.5))

анекдот о том что в ночь на поле с участием российских журналистов и не так как в мае 2011 года в игру после этого года в результате чего только в большинстве
анекдот про то время как выяснилось что в любом случае если бы не может быть не будет в дальнейшем число людей которые находятся в этом году даже не только что для
анекдот в целом ряде регионов россии по данным издания и не был не менее в январе этого года в этом случае если у нас в сша на месте преступления предусмотренного ч
анекдот а в связи с тем что не должны быть не было бы не собирается приобрести за игру под давлением главы мвд россии на данный момент в январе 2012 года в
анекдот о том что в ходе своего личного состава преступления предусмотренного ч 3 3 млрд рублей но в воскресенье в интернете читают новости и в этом году в компании и не


In [13]:
for i in range(5):
    print(lm.generate(start='анекдот', T=0.1))

анекдот про то что в том что в том что в том что в том что в том что в том что в том что в том что в том что
анекдот про то что в том что в том что в том что в том числе и в том что в том что в том что в том что в том
анекдот про то что в том что в этом году в россии в том что в том что в этом году в том что в том числе и не только в
анекдот про то что в том что в том что в том что в том что в том что в том что в том что в том что в том что
анекдот про то что в том что в этом году в этом году в том что в том что в том что в том что в том что в том что


In [14]:
for text in ds_test.sample(5).text:
    
    text_tokens = tokenize(text)
    
    text_tokens_shuffle = text_tokens.copy()
    random.shuffle(text_tokens_shuffle)

    print(lm.logprob(text_tokens, alpha=0.001), 'vs' ,lm.logprob(text_tokens_shuffle, alpha=0.001))

-4122.352548138131 vs -6873.669165808592
-1616.5254426505312 vs -2853.680749785033
-3711.364095554391 vs -6935.803972191169
-4573.072599378747 vs -8327.694175982702
-2611.9863312683583 vs -5336.482110632253
